##Transforming constructors data

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/3.silver_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
constructors_df = (
    spark
    .table(bronze_table)
    .filter(F.col("batch_id") == v_batch_id)
)


## Dropping the url column

In [0]:
constructors_valid_df = (
    constructors_df
    .select(
        F.col("constructorId"),
        F.col("name"),
        F.col("nationality"),
        F.col("ingestion_timestamp"),
        F.col("source_file"),
        F.col("batch_id")
    )
)

display(constructors_valid_df)

## Standardizing column names

In [0]:
constructors_renamed_df = (
    constructors_valid_df
    .withColumnsRenamed(
        {"constructorId": "constructor_id",
         "name": "constructor_name"}
    )
)

In [0]:
display(constructors_renamed_df)

## Dropping Columns with Business Key as Null

In [0]:
constructors_not_null_df = (
    constructors_renamed_df
    .filter(F.col("constructor_id").isNotNull())
)

display(constructors_not_null_df)

In [0]:
constructors_filtered_df = (
    constructors_not_null_df
    .dropDuplicates(["constructor_id"])
)

display(constructors_filtered_df)


Standardizing the colum values within `nationality`

In [0]:
constructors_final_df = (
    constructors_filtered_df
    .withColumns(
        {
            #"constructor_id": F.initcap(F.col("constructor_id")),
            #"constructor_name": F.initcap(F.col("constructor_name")),
            "nationality": F.initcap(F.col("nationality"))
        }
    )
)

display(constructors_final_df)

## Writing into the silver delta table

In [0]:
constructor_columns_to_update = [
    "constructor_id",
    "constructor_name",
    "nationality",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(
    constructors_final_df, 
    silver_table,
    merge_condition = "t.constructor_id = s.constructor_id",
    columns_to_update = constructor_columns_to_update
    )

In [0]:
spark.table(silver_table).display()